# Module 09 — Notebook 2: Metrics and Baselines

## Learning Objectives

By the end of this notebook, you will be able to:

- Compute accuracy, precision, recall, and F1 from scratch using Python dicts and lists
- Explain why you always need a baseline before interpreting a metric
- Compute a random baseline and a majority-class baseline
- Understand conceptually the difference between macro and micro averaging

**Estimated time:** ~25 minutes

## Why This Matters for AI Research Engineering

"Our model achieves 90% accuracy!" — sounds great, right? Not if 90% of the test examples have the same label. A model that always predicts the majority class would score 90% without learning anything.

Metrics without baselines are almost meaningless. At safety labs, you need to know not just *what score* your eval produces, but *how much better than chance* your model actually is. This notebook builds those skills from scratch — no sklearn, just Python.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_length
print("Setup complete.")

## 1. The Confusion Matrix

For binary classification (e.g., flagged/not-flagged), all four outcomes fit into a 2x2 table:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | TP (True Positive) | FN (False Negative) |
| **Actually Negative** | FP (False Positive) | TN (True Negative) |

In safety eval terms:
- **TP:** Model output was actually harmful AND we flagged it — correct catch
- **FP:** Model output was actually fine BUT we flagged it — false alarm
- **FN:** Model output was actually harmful BUT we missed it — dangerous miss
- **TN:** Model output was actually fine AND we didn't flag it — correct pass

All four metrics (accuracy, precision, recall, F1) are computed from these four numbers.

In [ ]:
# A concrete confusion matrix for a safety classifier
# Scenario: a classifier that flags potentially harmful model outputs

TP = 18  # correctly flagged harmful outputs
FP = 5   # clean outputs incorrectly flagged
FN = 7   # harmful outputs that slipped through
TN = 70  # clean outputs correctly passed

total = TP + FP + FN + TN
print(f"Total examples: {total}")
print(f"  TP={TP}, FP={FP}, FN={FN}, TN={TN}")

## 2. Accuracy

**Accuracy** = fraction of examples that were classified correctly.

```
accuracy = (TP + TN) / (TP + FP + FN + TN)
```

Simple and intuitive — but misleading on imbalanced datasets. If 95% of outputs are clean, a classifier that always says "clean" gets 95% accuracy while catching *zero* harmful outputs.

In [ ]:
def compute_accuracy(tp, fp, fn, tn):
    """Compute accuracy from confusion matrix values."""
    total = tp + fp + fn + tn
    return (tp + tn) / total

accuracy = compute_accuracy(TP, FP, FN, TN)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")

## 3. Precision and Recall

For safety classifiers, accuracy alone misses what matters. We care about:

**Precision** = of everything we flagged, how much was actually harmful?
```
precision = TP / (TP + FP)
```
High precision → few false alarms. Low precision → lots of unnecessary flags.

**Recall** = of everything actually harmful, how much did we catch?
```
recall = TP / (TP + FN)
```
High recall → we catch most harmful content. Low recall → dangerous misses.

**The tradeoff:** raising your threshold makes precision go up but recall go down (you flag less, but what you flag is more reliable). Safety contexts often prioritize recall — missing a harmful output is worse than a false alarm.

In [ ]:
def compute_precision(tp, fp):
    """Precision: of what we flagged, how much is actually positive?"""
    if tp + fp == 0:
        return 0.0  # avoid division by zero
    return tp / (tp + fp)

def compute_recall(tp, fn):
    """Recall: of all positives, how many did we catch?"""
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

precision = compute_precision(TP, FP)
recall = compute_recall(TP, FN)

print(f"Precision: {precision:.4f} ({precision*100:.1f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.1f}%)")
print()
print("Interpretation:")
print(f"  {precision*100:.0f}% of our flags were genuinely harmful")
print(f"  We caught {recall*100:.0f}% of all harmful outputs")

## 4. F1 Score

**F1** is the harmonic mean of precision and recall. It gives you a single number that balances both:

```
F1 = 2 * (precision * recall) / (precision + recall)
```

Why harmonic mean? Because it punishes extreme imbalances. A classifier with 100% precision and 0% recall gets F1 = 0 (it didn't catch anything). A classifier with 0% precision and 100% recall also gets F1 = 0 (it flagged everything, catching nothing useful).

F1 is the go-to metric when you have class imbalance and both precision and recall matter.

In [ ]:
def compute_f1(precision, recall):
    """F1 score: harmonic mean of precision and recall."""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

f1 = compute_f1(precision, recall)
print(f"F1 Score: {f1:.4f}")
print()

# Show what happens at extremes
print("F1 at extremes:")
print(f"  High precision (0.95), low recall (0.10): F1 = {compute_f1(0.95, 0.10):.3f}")
print(f"  Low precision (0.10), high recall (0.95): F1 = {compute_f1(0.10, 0.95):.3f}")
print(f"  Balanced (0.80, 0.80):                   F1 = {compute_f1(0.80, 0.80):.3f}")

## 5. Why You Always Need a Baseline

A metric score is only meaningful relative to a reference point. Two common baselines:

**Random baseline:** what score would a classifier get if it assigned labels randomly? For balanced classes, this is 50% accuracy. For imbalanced classes, it depends on the class distribution.

**Majority-class baseline:** what score would a classifier get if it *always predicted the most common class*? This is the simplest possible model — if your classifier can't beat it, something is very wrong.

In JS terms: these are like having a function that always returns `null` or `true` — your actual model needs to beat those trivial implementations to justify its complexity.

In [ ]:
# Simulated labels from our synthetic dataset
# 1 = flagged (positive class), 0 = not flagged
actual_labels = [0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1]

total = len(actual_labels)
n_positive = sum(actual_labels)          # count of 1s
n_negative = total - n_positive          # count of 0s

print(f"Dataset: {total} examples")
print(f"  Positive (flagged): {n_positive} ({n_positive/total*100:.1f}%)")
print(f"  Negative (clean):   {n_negative} ({n_negative/total*100:.1f}%)")
print()

# Majority class baseline: always predict 0 (the majority)
majority_class = 0 if n_negative >= n_positive else 1
majority_predictions = [majority_class] * total
majority_correct = sum(a == p for a, p in zip(actual_labels, majority_predictions))
majority_accuracy = majority_correct / total

print(f"Majority class: {majority_class}")
print(f"Majority-class baseline accuracy: {majority_accuracy:.4f} ({majority_accuracy*100:.1f}%)")

## 6. Macro vs Micro Averaging (Conceptual)

When you have more than two classes, you need to decide how to average metrics across classes.

**Macro averaging:** compute the metric for each class separately, then take the unweighted mean. Every class counts equally, regardless of how many examples it has. This is sensitive to performance on rare classes.

**Micro averaging:** pool all TP, FP, FN counts across classes, then compute the metric once. Rare classes get less weight because they contribute fewer examples.

| Averaging | When to use |
|---|---|
| Macro | When you care about each category equally (e.g., each harm type matters, regardless of frequency) |
| Micro | When you care about overall performance across all examples |

For binary safety evals, you're usually computing one-class metrics directly, so averaging is less of a concern.

In [ ]:
# Conceptual demonstration: per-class precision for 3 categories
per_class_precision = {
    "harmful_content":  0.90,  # 90 examples
    "misinformation":   0.75,  # 30 examples
    "factual_error":    0.60   # 5 examples
}

per_class_counts = {
    "harmful_content":  90,
    "misinformation":   30,
    "factual_error":    5
}

# Macro average: simple mean
macro_precision = sum(per_class_precision.values()) / len(per_class_precision)

# Micro average: weighted by class size
total_examples = sum(per_class_counts.values())
micro_precision = sum(
    per_class_precision[cls] * per_class_counts[cls]
    for cls in per_class_precision
) / total_examples

print(f"Macro precision: {macro_precision:.4f}")
print(f"Micro precision: {micro_precision:.4f}")
print()
print("Notice: micro is pulled toward the large 'harmful_content' class (0.90)")
print("        macro gives equal weight to the rare 'factual_error' class (0.60)")

## Exercise 1 — Compute Precision, Recall, F1

A new classifier has the following confusion matrix values:

```python
TP2 = 30
FP2 = 12
FN2 = 3
TN2 = 55
```

Using the functions defined above (`compute_precision`, `compute_recall`, `compute_f1`), compute:
- `precision2` — rounded to 4 decimal places
- `recall2` — rounded to 4 decimal places
- `f1_2` — rounded to 4 decimal places

In [ ]:
TP2 = 30
FP2 = 12
FN2 = 3
TN2 = 55

# YOUR CODE HERE
precision2 = None   # float, rounded to 4 decimal places
recall2 = None      # float, rounded to 4 decimal places
f1_2 = None         # float, rounded to 4 decimal places

In [ ]:
check_type(precision2, float, "precision2 is a float")
check_type(recall2, float, "recall2 is a float")
check_type(f1_2, float, "f1_2 is a float")
check_approx(precision2, 0.7143, 0.001, "precision2 correct")
check_approx(recall2, 0.9091, 0.001, "recall2 correct")
check_approx(f1_2, 0.8000, 0.001, "f1_2 correct")

## Exercise 2 — Compute Majority-Class Baseline Accuracy

Given a list of true labels called `labels_2`, compute the majority-class baseline accuracy. Store the result in `baseline_acc` as a float rounded to 4 decimal places.

Steps:
1. Count how many times each label (0 or 1) appears
2. Find the majority class (the one that appears most)
3. Compute the fraction of examples that have the majority label

In [ ]:
labels_2 = [0, 1, 0, 0, 1, 0, 0, 0, 1, 0,
            0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
            1, 0, 0, 0, 0, 1, 0, 0, 0, 0]

# YOUR CODE HERE
baseline_acc = None  # float, rounded to 4 decimal places

In [ ]:
check_type(baseline_acc, float, "baseline_acc is a float")
check_approx(baseline_acc, 0.7667, 0.001, "majority-class baseline accuracy correct")

## Exercise 3 — Compare Classifier to Baseline

Use `labels_2` from Exercise 2 and a set of model predictions. Compute:
1. The actual classifier accuracy (stored in `classifier_acc`, float, 4 decimal places)
2. The improvement over the majority-class baseline (stored in `improvement`, float, 4 decimal places)

Improvement = `classifier_acc - baseline_acc`

In [ ]:
# Predictions from a model
predictions_2 = [0, 1, 0, 0, 1, 0, 0, 0, 1, 0,
                 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
                 1, 0, 0, 0, 0, 1, 0, 0, 1, 0]

# YOUR CODE HERE
classifier_acc = None   # float, rounded to 4 decimal places
improvement = None      # float, rounded to 4 decimal places

In [ ]:
check_type(classifier_acc, float, "classifier_acc is a float")
check_type(improvement, float, "improvement is a float")
check_approx(classifier_acc, 0.9333, 0.001, "classifier accuracy correct")
check_approx(improvement, 0.1667, 0.001, "improvement over baseline correct")

## Wrap-Up

| Metric | Formula | Use when |
|---|---|---|
| **Accuracy** | (TP + TN) / total | Balanced classes; quick overview |
| **Precision** | TP / (TP + FP) | False alarms are costly |
| **Recall** | TP / (TP + FN) | Missing positives is costly (safety!) |
| **F1** | 2 * P * R / (P + R) | Imbalanced classes; need to balance P and R |
| **Majority baseline** | most-common-class fraction | Always compute this before claiming your model works |
| **Macro avg** | mean of per-class metrics | Equal weight to each category |
| **Micro avg** | pooled TP/FP/FN | Equal weight to each example |

**Key rule:** always compare your model's metric to the majority-class baseline. If the improvement is small, re-examine your model and your eval design.

**Next:** Notebook 3 — Inter-Rater Agreement: what to do when humans disagree about labels.